In [30]:
import numpy as np

Initializing needed parameters.

In [31]:
V52diameter = 52
V52height = 44  # hub height

V27diameter = 27
V27height = 31.5 # hub height

mast_height = V52height

V52position = np.array([317548, 6174985])
V27position = np.array([317519, 6174901])
mast_position = np.array([317438, 6175028])

Reference measurements to state whether an obstacle is relevant or not

In [32]:
(V52height - V52diameter) * 2 / 3 # reference to see if a nearby operating WT is small
def calculate_distance_coordinates(coord1, coord2):
    return np.sqrt((coord2[0] - coord1[0]) ** 2 + (coord2[1] - coord1[1]) ** 2)

L1 = calculate_distance_coordinates(V52position, mast_position) # distance between the mast and the V52
L2 = calculate_distance_coordinates(V27position, mast_position) # distance between the mast and the V27
L3 = calculate_distance_coordinates(V27position, V52position) # distance between the WTs
L1, L2, L3


(118.10588469674151, 150.63200191194434, 88.86506625215557)

Azimuthal positions with respect to North

In [33]:
dir1 = np.degrees(np.arctan((np.abs(V52position[1] - mast_position[1])) / (np.abs(V52position[0] - mast_position[0])))) + 90 # direction from the mast to the V52
dir2 = np.degrees(np.arctan((np.abs(V27position[1] - mast_position[1])) / (np.abs(V27position[0] - mast_position[0])))) + 90  # direction from the mast to the V27
dir3 = np.degrees(np.arctan(V27position[1] - V52position[1]) / (V27position[0] - V52position[0])) + 180  # direction from the V52 to the V27
dir1, dir2, dir3

(111.3509802476536, 147.4704617216689, 183.07992895195957)

Function to calculate disturbance sectors

In [34]:
def calculate_disturbance_sector(distance, diameter): # use equivalent distance and diameter for obstacles different from a WT

    alfa = 1.3 * (np.rad2deg(np.arctan ((2.5 * diameter / distance) + 0.15))) + 10
    ratio = distance / diameter
    
    return alfa, ratio


Wake-free influence sectors

In [35]:
alfa1, ratio1 = calculate_disturbance_sector(L1, V52diameter) # disturbance sector of the V52 on the mast
alfa2, ratio2 = calculate_disturbance_sector(L2, V27diameter) # disturbance sector of the V27 on the mast
alfa3, ratio3 = calculate_disturbance_sector(L3, V27diameter) # disturbance sector of the V27 on the V52
results = {
    "V52 on mast": {"alfa": alfa1, "ratio": ratio1},
    "V27 on mast": {"alfa": alfa2, "ratio": ratio2},
    "V27 on V52": {"alfa": alfa3, "ratio": ratio3}
}
results

{'V52 on mast': {'alfa': 76.76279903096264, 'ratio': 2.2712670133988753},
 'V27 on mast': {'alfa': 50.14939254563706, 'ratio': 5.578963033775716},
 'V27 on V52': {'alfa': 64.97567826079508, 'ratio': 3.291298750079836}}

The following obstacles can be identified:
- Building 399
- Ecotron building
- Wind turbine prototype southwards
- Group of trees EN
- Woods ES

The masts in south direction (3 measurement masts) are not considered obstacles (ref Assignment 2 for reasoning).
Assumptions:
- Building heights: 399 is 8 m tall, Ecotron is 10 m tall
- WT prototype is 20 m tall 
- Trees heights are 4 m

Initialize obstacles distances (on WT V52) and heights

In [36]:
obstacles = ["trees North-East (NE)", "build. 399", "Ecotron", "trees ES", "WT prototype"]
distances = np.array([153.08, 61.38, 78.62, 214.1, 282.11]) # respectively: trees North-East (NE), build. 399, Ecotron, trees ES, WT prototype
heights = np.array([4, 8, 10, 4, 20])


Reference distances and heights 

In [37]:
ref_distance = np.array([2*L1, 4*L1, 8*L1, 16*L1])
ref_heights = np.array([
    (V52height - V52diameter/2) / 3,
    (V52height - V52diameter/2) / 3 * 2,
    (V52height - V52diameter/2),
    (V52height - V52diameter/2) * 4 / 3
])

Comparison with reference measurements to check whether obstacles can be considered irrelevant or not

In [38]:
relevant_obstacles = []
for i in range(len(distances)):
    if distances[i] < ref_distance[0]:
        if heights[i] >= ref_heights[0]:
            print(f"Obstacle {i} being {obstacles[i]} is relevant")
            relevant_obstacles.append(obstacles[i])
    elif distances[i] < ref_distance[1]:
        if heights[i] >= ref_heights[1]:
            print(f"Obstacle {i} being {obstacles[i]} is relevant")
            relevant_obstacles.append(obstacles[i])
    elif distances[i] < ref_distance[2]:
        if heights[i] >= ref_heights[2]:
            print(f"Obstacle {i} being {obstacles[i]} is relevant")
            relevant_obstacles.append(obstacles[i])
    elif distances[i] < ref_distance[3]:
        if heights[i] >= ref_heights[3]:
            print(f"Obstacle {i} being {obstacles[i]} is relevant")
            relevant_obstacles.append(obstacles[i])
    else:
        print(f"Obstacle {i} being {obstacles[i]} is not relevant because is further than {ref_distance[3]} m")

relevant_obstacles

Obstacle 1 being build. 399 is relevant
Obstacle 2 being Ecotron is relevant
Obstacle 4 being WT prototype is relevant


['build. 399', 'Ecotron', 'WT prototype']

Calculation of equivalent diameter for relevant obstacles, so building 399, Ecotron and WT prototype. Definition of obstacles width as seen from V52 is also implemented in this step.

In [39]:
relevant_width = np.array([15.17, 35.94, 2.28]) # respectively, widths of: building 399, Ecotron, WT prototype
eq_diameters = []

def calculate_equivalent_diameter(obs_height, obs_width):
    return 2 * obs_height * obs_width / (obs_height + obs_width)

for i in range(len(relevant_obstacles)):
    if relevant_obstacles[i] == "build. 399":
        eq_diameter = calculate_equivalent_diameter(heights[1], relevant_width[0])
        print(f"Equivalent diameter of building 399: {eq_diameter} m")
        eq_diameters.append(eq_diameter)
    elif relevant_obstacles[i] == "Ecotron":
        eq_diameter = calculate_equivalent_diameter(heights[2], relevant_width[1])
        print(f"Equivalent diameter of Ecotron: {eq_diameter} m")
        eq_diameters.append(eq_diameter)
    elif relevant_obstacles[i] == "WT prototype":
        eq_diameter = calculate_equivalent_diameter(heights[4], relevant_width[2])
        print(f"Equivalent diameter of WT prototype: {eq_diameter} m")
        eq_diameters.append(eq_diameter)

eq_diameters

Equivalent diameter of building 399: 10.475615019421666 m
Equivalent diameter of Ecotron: 15.6464954288202 m
Equivalent diameter of WT prototype: 4.093357271095152 m


[10.475615019421666, 15.6464954288202, 4.093357271095152]

Computation of disturbances sector due to relevant obstacles.

In [40]:
alfa4, ratio4 = calculate_disturbance_sector(distances[1], eq_diameters[0]) # disturbance sector of building 399 on V52
alfa5, ratio5 = calculate_disturbance_sector(distances[2], eq_diameters[1]) # disturbance sector of Ecotron on V52
alfa6, ratio6 = calculate_disturbance_sector(distances[4], eq_diameters[2]) # disturbance sector of WT prototype on V52

results_obstacles = {
    "building 399 on V52": {"alfa": alfa4, "ratio": ratio4},
    "Ecotron on V52": {"alfa": alfa5, "ratio": ratio5}, 
    "WT prototype on V52": {"alfa": alfa6, "ratio": ratio6}
}
results_obstacles

{'building 399 on V52': {'alfa': 48.962016671148525,
  'ratio': 5.859321852340146},
 'Ecotron on V52': {'alfa': 52.80183543734507, 'ratio': 5.024767390094603},
 'WT prototype on V52': {'alfa': 23.717350116858324,
  'ratio': 68.91897807017546}}

Definition of direction angles referred to V52 turbine to compute disturbances sector of relevant obstacles on V52 (directions taken as 'heading' from Google Earth project)

In [41]:
dir4 = 33.83
dir5 = 34.52
dir6 = 194.92
dir4, dir5, dir6

(33.83, 34.52, 194.92)